# State Design Pattern 

explained using the Document Workflow example (Draft → Moderation → Published).

#### The Concept
The State Pattern allows an object to alter its behavior when its internal state changes. The object will appear to change its class. Analogy: A Mobile Phone.
- **Unlocked State**: Pressing the power button turns off the screen.
- **Locked State**: Pressing the power button turns on the lock screen.
- The same button does completely different things depending on the phone's state.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we create a `State` interface. Every state (Draft, Moderation, Published) is a separate class. The Context (`Document`) delegates actions to the current state object.

In [3]:
from abc import ABC, abstractmethod

class State(ABC):
    @abstractmethod
    def publish(self, document): pass

    @abstractmethod
    def reject(self, document): pass

#### CONCRETE STATES

In [4]:
class Draft(State):
    def publish(self, document):
        print("Draft: Sent to moderation.")
        document.state = Moderation() # Transition

    def reject(self, document):
        print("Draft: Cannot reject a draft. Delete it instead.")

class Moderation(State):
    def publish(self, document):
        print("Moderation: Approved! Document is now live.")
        document.state = Published() # Transition

    def reject(self, document):
        print("Moderation: Rejected. Sending back to draft.")
        document.state = Draft() # Transition

class Published(State):
    def publish(self, document):
        print("Published: Already live. No action.")

    def reject(self, document):
        print("Published: Un-publishing content...")
        document.state = Draft() # Transition

#### THE CONTEXT (The Document)

In [5]:
class Document:
    def __init__(self):
        self.state = Draft() # Initial State

    def publish(self):
        self.state.publish(self)

    def reject(self):
        self.state.reject(self)

#### CLIENT CODE

In [6]:
def main():
    doc = Document()

    # 1. Draft -> Moderation
    doc.publish() 

    # 2. Moderation -> Published
    doc.publish()

    # 3. Published -> Draft (Recall)
    doc.reject()

if __name__ == "__main__":
    main()

Draft: Sent to moderation.
Moderation: Approved! Document is now live.
Published: Un-publishing content...


## The Pythonic Way (Class Switching)

In Python, objects are mutable enough that we can literally swap the class of an object at runtime using the `__class__` attribute. This eliminates the need for a separate `self.state` wrapper object. The Document becomes the State.

#### THE STATES (Behaving as the Object)

In [7]:
class Draft:
    def publish(self):
        print("📝 Draft: Submitted for review.")
        self.__class__ = Moderation # Magic Switch!

    def reject(self):
        print("📝 Draft: Can't reject. It's not submitted yet.")

class Moderation:
    def publish(self):
        print("👩‍⚖️ Moderation: Approved. Going Live.")
        self.__class__ = Published

    def reject(self):
        print("👩‍⚖️ Moderation: Rejected. Back to Draft.")
        self.__class__ = Draft

class Published:
    def publish(self):
        print("✅ Published: Already live.")

    def reject(self):
        print("✅ Published: Recalling post...")
        self.__class__ = Draft

#### THE CONTEXT (Initialized as Draft)

In [8]:
class Document(Draft):
    """
    The Document starts as a Draft.
    It doesn't need extra methods; it inherits them from the current class.
    """
    def __init__(self):
        # We can hold shared data here
        self.content = "My Article"

#### CLIENT CODE

In [9]:
def main():
    # 1. Start as Draft
    doc = Document()
    print(f"Current State: {doc.__class__.__name__}")
    
    # 2. Publish (Changes class to Moderation)
    doc.publish()
    print(f"Current State: {doc.__class__.__name__}")

    # 3. Publish (Changes class to Published)
    doc.publish()
    print(f"Current State: {doc.__class__.__name__}")

    # 4. Reject (Changes class back to Draft)
    doc.reject()
    print(f"Current State: {doc.__class__.__name__}")

if __name__ == "__main__":
    main()

Current State: Document
📝 Draft: Submitted for review.
Current State: Moderation
👩‍⚖️ Moderation: Approved. Going Live.
Current State: Published
✅ Published: Recalling post...
Current State: Draft


### Key Differences

| Feature        | Classic OOP                                                     | Pythonic (`__class__` swap)                                      |
|----------------|-----------------------------------------------------------------|------------------------------------------------------------------|
| **Structure**  | Context *has a* State.                                           | Context *is a* State (and changes it at runtime).                |
| **Boilerplate**| High — requires delegation methods (`self.state.action()`).     | Zero — method calls work directly.                               |
| **Mechanism**  | `self.state = NewState()`                                        | `self.__class__ = NewState`                                      |
| **Safety**     | High — enforces interface contracts.                             | Medium — developer must ensure all states share the same methods. |


# State Design Pattern 

explained using a complex, real-world example: E-Commerce Order Fulfillment System.

#### The Scenario: Order Lifecycle

An Order goes through a strict lifecycle: `New` → `Paid` → `Shipped` → `Delivered`. There are strict rules for every transition:
- You cannot `Cancel` an order once it is Shipped.
- You cannot `Ship` an order unless it is Paid.
- You cannot `Pay` for an order that is already Delivered.

Handling this with a massive `if-else` block (`if state == 'Paid' and action == 'Ship'...`) creates "Spaghetti Code". The State Pattern solves this by isolating behaviors.

## The Classic OOP Way (Java-Style)

We define an abstract `OrderState` class. We create separate classes for `NewState`, `PaidState`, `ShippedState`, etc. The Order context delegates calls to the current state object.

#### THE STATE INTERFACE

In [10]:
from abc import ABC, abstractmethod

class OrderState(ABC):
    @abstractmethod
    def pay(self, order): pass

    @abstractmethod
    def ship(self, order): pass

    @abstractmethod
    def deliver(self, order): pass

    @abstractmethod
    def cancel(self, order): pass

#### CONCRETE STATES

In [11]:
class NewState(OrderState):
    def pay(self, order):
        print("✅ Payment verified. Order is now PAID.")
        order.set_state(PaidState())

    def ship(self, order):
        print("❌ Error: Can't ship unpaid order.")

    def deliver(self, order):
        print("❌ Error: Can't deliver unpaid order.")

    def cancel(self, order):
        print("⚠️ Order Cancelled.")
        order.set_state(CancelledState())

class PaidState(OrderState):
    def pay(self, order):
        print("❌ Error: Order already paid.")

    def ship(self, order):
        print("✅ Tracking generated. Order SHIPPED.")
        order.set_state(ShippedState())

    def deliver(self, order):
        print("❌ Error: Must ship before delivering.")

    def cancel(self, order):
        print("⚠️ Payment refunded. Order Cancelled.")
        order.set_state(CancelledState())

class ShippedState(OrderState):
    def pay(self, order):
        print("❌ Error: Already paid & shipped.")

    def ship(self, order):
        print("❌ Error: Already shipped.")

    def deliver(self, order):
        print("✅ Package arrived. Order DELIVERED.")
        order.set_state(DeliveredState())

    def cancel(self, order):
        print("❌ Error: Cannot cancel order in transit!")

class DeliveredState(OrderState):
    def pay(self, order): print("❌ Error: Order finished.")
    def ship(self, order): print("❌ Error: Order finished.")
    def deliver(self, order): print("❌ Error: Already delivered.")
    def cancel(self, order): print("❌ Error: Return item instead of cancelling.")

class CancelledState(OrderState):
    def pay(self, order): print("❌ Error: Order is dead.")
    def ship(self, order): print("❌ Error: Order is dead.")
    def deliver(self, order): print("❌ Error: Order is dead.")
    def cancel(self, order): print("❌ Error: Already cancelled.")

#### THE CONTEXT (The Order)

In [12]:
class Order:
    def __init__(self):
        self._state = NewState() # Initial State

    def set_state(self, state: OrderState):
        self._state = state

    def pay(self): self._state.pay(self)
    def ship(self): self._state.ship(self)
    def deliver(self): self._state.deliver(self)
    def cancel(self): self._state.cancel(self)

#### CLIENT CODE

In [13]:
def main():
    order = Order()
    
    print("--- 1. Happy Path ---")
    order.pay()
    order.ship()
    order.deliver()

    print("\n--- 2. Illegal Action ---")
    order.cancel() # Can't cancel delivered order

if __name__ == "__main__":
    main()

--- 1. Happy Path ---
✅ Payment verified. Order is now PAID.
✅ Tracking generated. Order SHIPPED.
✅ Package arrived. Order DELIVERED.

--- 2. Illegal Action ---
❌ Error: Return item instead of cancelling.


## The Pythonic Way (Table-Driven State Machine)

In Python, writing 5 classes just to manage transitions is verbose. We can use a Table-Driven Approach. We define a dictionary that maps (`Current State, Action`) to (`Next State, Handler Function`). This centralizes the entire lifecycle logic in one readable data structure.

#### ACTION HANDLERS (Pure Functions)

In [14]:
def process_payment(order):
    print("✅ Payment processed.")
    return True

def ship_goods(order):
    print("✅ Goods handed to logistics.")
    return True

def deliver_package(order):
    print("✅ Customer signed for package.")
    return True

def refund_money(order):
    print("⚠️ Money refunded to customer.")
    return True

#### THE PYTHONIC CONTEXT

In [17]:
from typing import Dict, Tuple, Callable

class SmartOrder:
    def __init__(self):
        self.state = "NEW"
        
        # THE STATE TRANSITION TABLE
        # Key: (CurrentState, Action) -> Value: (NextState, Function)
        self.transitions: Dict[Tuple[str, str], Tuple[str, Callable]] = {
            ("NEW", "pay"):       ("PAID", process_payment),
            ("NEW", "cancel"):    ("CANCELLED", lambda o: print("Order dropped.")),
            
            ("PAID", "ship"):     ("SHIPPED", ship_goods),
            ("PAID", "cancel"):   ("CANCELLED", refund_money),
            
            ("SHIPPED", "deliver"): ("DELIVERED", deliver_package),
        }

    def trigger(self, action: str):
        # 1. Look up the transition
        key = (self.state, action)
        
        if key not in self.transitions:
            print(f"❌ Action '{action}' is invalid in state '{self.state}'")
            return

        # 2. Extract next state and logic
        next_state, handler = self.transitions[key]
        
        # 3. Execute logic
        success = handler(self)
        
        # 4. Update state only if logic succeeded
        if success:
            print(f"   [State Change]: {self.state} -> {next_state}")
            self.state = next_state

#### CLIENT CODE

In [18]:
def main():
    order = SmartOrder()

    print("--- 1. Normal Flow ---")
    order.trigger("pay")
    order.trigger("ship")
    
    print("\n--- 2. Trying Forbidden Action ---")
    # Cannot cancel after shipping (Rule is enforced by dictionary missing that key)
    order.trigger("cancel") 
    
    print("\n--- 3. Completion ---")
    order.trigger("deliver")

if __name__ == "__main__":
    main()

--- 1. Normal Flow ---
✅ Payment processed.
   [State Change]: NEW -> PAID
✅ Goods handed to logistics.
   [State Change]: PAID -> SHIPPED

--- 2. Trying Forbidden Action ---
❌ Action 'cancel' is invalid in state 'SHIPPED'

--- 3. Completion ---
✅ Customer signed for package.
   [State Change]: SHIPPED -> DELIVERED


#### Why the Pythonic version wins here

- **Centralized Logic**: In the OOP version, to see all rules about "Cancellation", you have to open 5 different files/classes. In the Pythonic version, you just look at the `self.transitions` dictionary. It acts as a configuration table.
- **Scalability**: Adding a new state (e.g., "Returned") is as simple as adding a new entry to the dictionary.
- **Strictness**: If a transition isn't in the dictionary, it's impossible. This prevents "illegal states" by default.